# Lost in the Museum - deskew + DINOv2 ViT-g/14 @ 518px

Identical to the run that scored **0.8285**, with one preprocessing step added:
each image is straightened before embedding.

### Why straighten instead of train

Every real query is a painting rotated a few degrees inside a pale frame
(16000, 17932, 11728, 05362 all show it). Two attempts to teach a model to
tolerate that rotation scored *worse* than zero-shot -- 0.68456 and 0.76845
against 0.79194 for the untouched backbone. Teaching invariance to a
transformation costs discriminative power, and this gallery holds 9,000 artwork
distractors where fine detail is exactly what separates one painting from
another.

Removing the rotation instead costs nothing: the query simply becomes a normal
upright image, and the domain gap it was creating disappears.

### Detection

The rotation is found by minimising the area of the content's axis-aligned
bounding box -- a tilted rectangle has a larger bounding box than an upright
one. There is no training and no labels, and it is a no-op on images that are
already straight, so the same routine runs safely over all 20,000.

Measured on 40 detected queries against the existing gallery embeddings, top-1
similarity rose from 0.850 to 0.861, improving for 27 of 40, with individual
gains as large as 0.838 -> 0.950.

### The risk, stated plainly

On a random sample the detector fires on 9.7% of images while queries are only
~5% of the corpus, so perhaps half the fires are on gallery images. Rotating an
already-upright gallery scan moves it *away* from its query. Whether the gain on
true queries outweighs that is what this run measures.

**Settings:** GPU ON, Internet ON. Roughly 1.5-2 hours.


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None


def find_data_dir():
    best, best_n = None, 0
    for d in Path("/kaggle/input").rglob("*"):
        if d.is_dir():
            n = sum(1 for _ in d.glob("*.png"))
            if n > best_n:
                best, best_n = d, n
    return best, best_n


DATA_DIR, _n = find_data_dir()
print("Data:", DATA_DIR, f"({_n} png)")
assert DATA_DIR is not None and _n == 20000, "expected 20000 images"

MODEL, SIZE, DIM, BATCH, FP16 = "dinov2_vitg14", 518, 1536, 4, True
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Enable Settings -> Accelerator -> GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"VRAM {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")

## Deskewing


In [ ]:
Image.MAX_IMAGE_PIXELS = None

WORK_SIZE = 220          # detection resolution; the angle does not need full detail
ANGLE_LIMIT = 16.0
COARSE_STEP = 2.0
FINE_STEP = 0.25
MIN_GAIN = 0.12          # only act on clearly rotated images.
#   A loose threshold would also rotate upright gallery scans, whose bounding
#   box shrinks slightly at any angle through interpolation noise. Those are
#   90% of the corpus, so a false positive there costs more than a missed query.


def _content_mask(img: Image.Image, tol: int = 18) -> np.ndarray:
    """True where the image differs from its padding colour.

    The padding colour is taken from the corners rather than assumed: different
    queries use slightly different pale values.
    """
    small = img.convert("RGB")
    small.thumbnail((WORK_SIZE, WORK_SIZE))
    a = np.asarray(small).astype(np.int16)
    h, w, _ = a.shape
    k = max(3, min(h, w) // 12)
    corners = np.concatenate([
        a[:k, :k].reshape(-1, 3), a[:k, -k:].reshape(-1, 3),
        a[-k:, :k].reshape(-1, 3), a[-k:, -k:].reshape(-1, 3),
    ])
    fill = np.median(corners, axis=0)
    return (np.abs(a - fill).sum(axis=2) > tol)


def _bbox_area(mask: np.ndarray) -> float:
    rows = np.flatnonzero(mask.any(axis=1))
    cols = np.flatnonzero(mask.any(axis=0))
    if len(rows) == 0 or len(cols) == 0:
        return float(mask.size)
    return float((rows[-1] - rows[0] + 1) * (cols[-1] - cols[0] + 1))


def _rotate_mask(mask: np.ndarray, deg: float) -> np.ndarray:
    im = Image.fromarray((mask * 255).astype(np.uint8))
    out = im.rotate(deg, resample=Image.NEAREST, expand=True, fillcolor=0)
    return np.asarray(out) > 127


def estimate_angle(img: Image.Image) -> tuple[float, float]:
    """Rotation to undo, and the fractional bounding-box shrink it achieves."""
    mask = _content_mask(img)
    if mask.mean() < 0.05 or mask.mean() > 0.995:
        return 0.0, 0.0                      # no padding to work with
    base = _bbox_area(mask)

    def area(deg: float) -> float:
        return _bbox_area(_rotate_mask(mask, deg))

    coarse = np.arange(-ANGLE_LIMIT, ANGLE_LIMIT + COARSE_STEP, COARSE_STEP)
    best = min(coarse, key=area)
    fine = np.arange(best - COARSE_STEP, best + COARSE_STEP + FINE_STEP, FINE_STEP)
    best = min(fine, key=area)
    gain = (base - area(best)) / base
    return (float(best), float(gain)) if gain > MIN_GAIN else (0.0, float(gain))


def deskew(img: Image.Image, tol: int = 18) -> tuple[Image.Image, float]:
    """Rotate the painting upright and crop away the padding.

    Returns the corrected image and the angle removed (0.0 if left untouched).
    """
    deg, gain = estimate_angle(img)
    if deg == 0.0:
        return img, 0.0

    corners = np.asarray(img.convert("RGB").resize((32, 32)))
    fill = tuple(int(v) for v in np.median(
        np.concatenate([corners[:6, :6].reshape(-1, 3), corners[:6, -6:].reshape(-1, 3),
                        corners[-6:, :6].reshape(-1, 3), corners[-6:, -6:].reshape(-1, 3)]),
        axis=0))

    rotated = img.convert("RGB").rotate(deg, resample=Image.BICUBIC, expand=True,
                                        fillcolor=fill)
    mask = _content_mask(rotated, tol)
    rows = np.flatnonzero(mask.any(axis=1))
    cols = np.flatnonzero(mask.any(axis=0))
    if len(rows) == 0 or len(cols) == 0:
        return rotated, deg
    sy = rotated.height / mask.shape[0]
    sx = rotated.width / mask.shape[1]
    box = (int(cols[0] * sx), int(rows[0] * sy),
           int((cols[-1] + 1) * sx), int((rows[-1] + 1) * sy))
    if box[2] - box[0] < 16 or box[3] - box[1] < 16:
        return rotated, deg
    return rotated.crop(box), deg

## Dataset

Straightening runs inside the DataLoader workers, so it overlaps with GPU work
and costs little wall-clock time.


In [ ]:
class ImageFolder(Dataset):
    def __init__(self, paths, size):
        self.paths, self.size = paths, size
        self.tf = transforms.Compose([
            transforms.Resize((size, size),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
            fixed, deg = deskew(img)
            return self.tf(fixed), i, float(deg != 0.0)
        except Exception as e:
            print(f"  ! failed {self.paths[i].name}: {e}")
            return torch.zeros(3, self.size, self.size), i, 0.0   # never drop a row


def gem_pool(patch_tokens, p=3.0, eps=1e-6):
    return patch_tokens.clamp(min=eps).pow(p).mean(dim=1).pow(1.0 / p)

## Embed


In [ ]:
paths = sorted(DATA_DIR.glob("*.png"))
model = torch.hub.load("facebookresearch/dinov2", MODEL, verbose=False).eval().to(device)
if FP16:
    model = model.half()

loader = DataLoader(ImageFolder(paths, SIZE), batch_size=BATCH, shuffle=False,
                    num_workers=4, pin_memory=True)

feats, n_fixed, t0 = None, 0.0, time.time()
with torch.no_grad():
    for batch, idxs, was_fixed in loader:
        batch = batch.to(device, non_blocking=True)
        if FP16:
            batch = batch.half()
        out = model.forward_features(batch)
        vec = torch.cat([
            F.normalize(out["x_norm_clstoken"], dim=1),
            F.normalize(gem_pool(out["x_norm_patchtokens"]), dim=1),
        ], dim=1).float().cpu().numpy()
        if feats is None:
            feats = np.zeros((len(paths), vec.shape[1]), dtype=np.float32)
        feats[idxs.numpy()] = vec
        n_fixed += float(was_fixed.sum())
        done = int(idxs[-1]) + 1
        if done % (BATCH * 200) < BATCH:
            r = done / (time.time() - t0)
            print(f"  {done}/{len(paths)}  {r:.1f} img/s  "
                  f"ETA {(len(paths)-done)/r/60:.0f} min  deskewed so far {int(n_fixed)}",
                  flush=True)

print(f"embedded {feats.shape} in {(time.time()-t0)/60:.1f} min")
print(f"straightened {int(n_fixed)}/{len(paths)} images ({100*n_fixed/len(paths):.1f}%)")
np.save("/kaggle/working/features_deskew.npy", feats)

## Whitened PCA and submission


In [ ]:
def l2(x, eps=1e-12):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


x = l2(feats.astype(np.float64))
mu = x.mean(axis=0, keepdims=True)
_, s, vt = np.linalg.svd(x - mu, full_matrices=False)
dim = min(DIM, x.shape[1])
x = l2((x - mu) @ vt[:dim].T / (s[:dim] / np.sqrt(len(x) - 1) + 1e-8)).astype(np.float32)
print(f"PCA {feats.shape[1]} -> {dim}  explains {(s[:dim]**2).sum()/(s**2).sum():.1%}")

df = pd.DataFrame(x, columns=[f"feature_{i}" for i in range(x.shape[1])])
df.insert(0, "image_name", [p.name for p in paths])
df["ID"] = df["image_name"]
df.to_csv("/kaggle/working/submission.csv", index=False, float_format="%.6f")
print(f"rows={len(df)}  cols={df.shape[1]}")
df.iloc[:3, :5]